# Benchmark run analysis

Starter notebook for poking at a `benchmark/<run>/runs.tsv` produced by `phlag.benchmark`.

In [1]:
import pandas as pd
import numpy as np

from phlag.utils import get_data_dir, get_phlag_output_base

In [2]:
DIST_TYPE = "gaussian"
WINDOW = "50k"
STEP = "1k"
RUN_A = 4
RUN_B = 10

benchmark_dir = (
    get_phlag_output_base(get_data_dir()) / DIST_TYPE / f"w{WINDOW}_s{STEP}" / "benchmark"
)

run_dir_a = benchmark_dir / str(RUN_A)
run_dir_b = benchmark_dir / str(RUN_B)

runs_path_a = run_dir_a / "runs.tsv"
runs_path_b = run_dir_b / "runs.tsv"

In [3]:
df_a = pd.read_csv(runs_path_a, sep="\t")
df_b = pd.read_csv(runs_path_b, sep="\t")

Drop columns that aren't run metrics: filesystem paths, panel-membership flags (`benchmark`'s own figure-inclusion bookkeeping), and the run/leaf identifiers (kept as the index instead of a metric column).

In [4]:
PATH_COLUMNS = ["report_path"]
PANEL_COLUMNS = ["in_panel_a", "in_panel_b", "in_panel_relerr", "in_panel_em_divergence", "anomaly_fraction_source"]
ID_COLUMNS = ["run_id", "source_leaf", 'x_variable']

df_a = df_a.drop(columns=PATH_COLUMNS + PANEL_COLUMNS+ID_COLUMNS, errors="ignore")
df_b = df_b.drop(columns=PATH_COLUMNS + PANEL_COLUMNS+ID_COLUMNS, errors="ignore")

In [5]:
df_b.columns

Index(['category', 'subcategory', 'column', 'pattern', 'anomaly_fraction',
       'fraction_bin', 'x_value', 'x_bin', 'branch_length_cu', 'clade_name',
       'clade_number', 'tpr', 'fpr', 'f1', 'accuracy', 'tp', 'fp', 'fn', 'tn',
       'n_windows', 'label_flipped', 'mean_relerr_agg', 'covar_relerr_agg',
       'em_bd', 'em_gt_bd', 'transition_null_to_null',
       'transition_null_to_alt', 'transition_alt_to_null',
       'transition_alt_to_alt', 'bic', 'log_likelihood', 'n_trainable_params',
       'relerr_ABBA_Null_mean', 'relerr_ABBA_Null_std', 'relerr_ABBA_Alt_mean',
       'relerr_ABBA_Alt_std', 'relerr_BABA_Null_mean', 'relerr_BABA_Null_std',
       'relerr_BABA_Alt_mean', 'relerr_BABA_Alt_std', 'relerr_AABB_Null_mean',
       'relerr_AABB_Null_std', 'relerr_AABB_Alt_mean', 'relerr_AABB_Alt_std',
       'exclusion'],
      dtype='object')

In [6]:
df_b['n_windows'].unique()

array([5951])

In [7]:
total = df_b['n_windows'][0]
alt_a = df_a['fp'] + df_a['tp']
alt_b = df_b['fp'] + df_b['tp']
print(np.mean(alt_a / total), np.mean(alt_b / total))

0.3999617911355435 0.3827420461547079


In [8]:
filter = 'x_bin'
target = 'f1'
value1 = "(0,0.1]"
value2 = "(0.1,0.25]"
low_drop = np.mean(df_b[df_b[filter]==value1][target] - df_a[df_a[filter]==value1][target])
high_drop = np.mean(df_b[df_b[filter]==value2][target] - df_a[df_a[filter]==value2][target])
print(low_drop, high_drop)

0.02621410256410256 nan
